# Task 04 – Dimensionality Reduction & Unsupervised Clustering

This notebook demonstrates:
- Numerical feature standardization
- Principal Component Analysis (PCA) and explained-variance scree plot
- K-Means clustering with the Elbow Method and Silhouette Score
- DBSCAN clustering
- Hierarchical clustering
- 2D cluster visualizations

**Dataset:** If the Academy provides a CSV dataset, place it in the same folder and set `USE_ACADEMY_CSV = True` in the next cell. Otherwise, this notebook uses the built-in Breast Cancer Wisconsin dataset from scikit-learn as a reproducible scientific dataset. Use the Academy-provided dataset if your task instructions specify one.


## 1. Install and import libraries

In [ ]:
# If packages are missing, uncomment and run:
# %pip install pandas numpy matplotlib seaborn scikit-learn scipy

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
sns.set_theme()


## 2. Load dataset
If you have the Academy CSV, set the flag to True and enter its filename. The target/label column, if any, is excluded from clustering.

In [ ]:
USE_ACADEMY_CSV = False
CSV_FILE = "your_dataset.csv"
LABEL_COLUMN = None  # Example: "target"; leave None if there is no label column

if USE_ACADEMY_CSV:
    raw_df = pd.read_csv(CSV_FILE)
    if LABEL_COLUMN and LABEL_COLUMN in raw_df.columns:
        raw_df = raw_df.drop(columns=[LABEL_COLUMN])
    df = raw_df.select_dtypes(include=[np.number]).copy()
else:
    dataset = load_breast_cancer(as_frame=True)
    df = dataset.data.copy()

print("Dataset shape:", df.shape)
display(df.head())
print("\nNumeric columns:", list(df.columns))


## 3. Clean and standardize numerical features

In [ ]:
# Replace infinite values and impute missing numeric values with column medians
X_df = df.replace([np.inf, -np.inf], np.nan).copy()
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X_df)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("Number of samples:", X_scaled.shape[0])
print("Number of numeric features:", X_scaled.shape[1])
print("Standardization complete.")


## 4. Principal Component Analysis (PCA) and scree plot

In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(explained) + 1), explained, marker="o", label="Individual explained variance")
plt.plot(range(1, len(cumulative) + 1), cumulative, marker="s", label="Cumulative explained variance")
plt.axhline(0.90, linestyle="--", label="90% cumulative variance")
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title("PCA Scree Plot")
plt.legend()
plt.grid(True)
plt.show()

n_components_90 = int(np.argmax(cumulative >= 0.90) + 1) if np.any(cumulative >= 0.90) else len(explained)
print(f"Components needed to explain at least 90% of variance: {n_components_90}")


## 5. Select the number of clusters using Elbow and Silhouette methods

In [ ]:
# Use first two PCA components for a compact clustering demonstration
n_pca = min(2, X_pca_full.shape[1])
X_pca = X_pca_full[:, :n_pca]

if len(X_pca) < 3:
    raise ValueError("At least 3 rows are required for clustering and silhouette analysis.")

max_k = min(10, len(X_pca) - 1)
k_values = list(range(2, max_k + 1))
inertias, silhouettes = [], []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(k_values, inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].grid(True)

axes[1].plot(k_values, silhouettes, marker="o")
axes[1].set_title("Silhouette Score by k")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")
axes[1].grid(True)
plt.tight_layout()
plt.show()

best_k = k_values[int(np.argmax(silhouettes))]
print("Best k by silhouette score:", best_k)
print("Best silhouette score:", round(max(silhouettes), 4))


## 6. K-Means clustering and 2D visualization

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_pca)
kmeans_silhouette = silhouette_score(X_pca, kmeans_labels)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1] if X_pca.shape[1] > 1 else np.zeros(len(X_pca)),
                hue=kmeans_labels, palette="tab10", s=45)
plt.title(f"K-Means Clusters (k={best_k}), Silhouette={kmeans_silhouette:.3f}")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.show()

print("K-Means silhouette score:", round(kmeans_silhouette, 4))


## 7. DBSCAN clustering

In [ ]:
# DBSCAN can identify noise points as cluster -1.
# These values are starting settings; tune eps/min_samples for your dataset.
dbscan = DBSCAN(eps=1.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_pca)

unique_dbscan = set(dbscan_labels)
n_dbscan_clusters = len(unique_dbscan - {-1})
n_noise = int(np.sum(dbscan_labels == -1))

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1] if X_pca.shape[1] > 1 else np.zeros(len(X_pca)),
                hue=dbscan_labels, palette="tab10", s=45)
plt.title("DBSCAN Clustering (noise labeled -1)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.show()

print("DBSCAN clusters (excluding noise):", n_dbscan_clusters)
print("DBSCAN noise points:", n_noise)
if n_dbscan_clusters >= 2:
    non_noise = dbscan_labels != -1
    print("DBSCAN silhouette score (excluding noise):",
          round(silhouette_score(X_pca[non_noise], dbscan_labels[non_noise]), 4))
else:
    print("Silhouette score not calculated: DBSCAN found fewer than 2 non-noise clusters.")


## 8. Hierarchical (Agglomerative) clustering

In [ ]:
hierarchical = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
hier_labels = hierarchical.fit_predict(X_pca)
hier_silhouette = silhouette_score(X_pca, hier_labels)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1] if X_pca.shape[1] > 1 else np.zeros(len(X_pca)),
                hue=hier_labels, palette="tab10", s=45)
plt.title(f"Hierarchical Clusters (k={best_k}), Silhouette={hier_silhouette:.3f}")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.show()

print("Hierarchical silhouette score:", round(hier_silhouette, 4))


## 9. Compare clustering methods

In [ ]:
comparison = pd.DataFrame({
    "Method": ["K-Means", "Hierarchical"],
    "Number of clusters": [len(set(kmeans_labels)), len(set(hier_labels))],
    "Silhouette score": [kmeans_silhouette, hier_silhouette]
})
display(comparison)

print("DBSCAN cluster count (excluding noise):", n_dbscan_clusters)
print("DBSCAN noise points:", n_noise)


## 10. Save results and conclusion

In [ ]:
results = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1] if X_pca.shape[1] > 1 else np.zeros(len(X_pca)),
    "KMeans_Cluster": kmeans_labels,
    "DBSCAN_Cluster": dbscan_labels,
    "Hierarchical_Cluster": hier_labels
})
results.to_csv("clustering_results.csv", index=False)
print("Saved clustering_results.csv")


### Conclusion

- Numerical features were cleaned, imputed, and standardized.
- PCA reduced the feature space and the scree plot showed explained variance.
- The Elbow Method and Silhouette Score were used to assess the number of K-Means clusters.
- K-Means, DBSCAN, and Hierarchical clustering were applied and visualized in PCA space.
- Cluster labels and PCA coordinates were exported to `clustering_results.csv`.

**Before submission:** Run every cell using the Academy's required dataset if one is specified. Review the plots and scores, then save the notebook with its outputs. The built-in dataset is only a fallback demonstration, not a replacement for an Academy-mandated dataset.
